In [ ]:
import requests
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, ArrayType
)
from pyspark.sql.functions import to_timestamp, col, explode

In [2]:
url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&topics=manufacturing,technology&apikey={os.getenv('ALPHA_VANTAGE_API_KEY')}"
r = requests.get(url)
data = r.json()

In [3]:
# Inicia a SparkSession (local)
#spark = SparkSession.builder \
#    .appName("AlphaVantageNewsSentiment") \
#    .master("local[*]") \
#    .getOrCreate()

# --- Sub-schemas ---

topic_schema = StructType([
    StructField("topic",           StringType(), True),
    StructField("relevance_score", FloatType(),  True),
])

ticker_schema = StructType([
    StructField("ticker",                 StringType(), True),
    StructField("relevance_score",        FloatType(),  True),
    StructField("ticker_sentiment_score", FloatType(),  True),
    StructField("ticker_sentiment_label", StringType(), True),
])

# --- Schema principal ---

schema = StructType([
    StructField("title",                   StringType(),              True),
    StructField("url",                     StringType(),              True),
    StructField("time_published",          StringType(),              True),
    StructField("authors",                 StringType(),              True),
    StructField("source",                  StringType(),              True),
    StructField("source_domain",           StringType(),              True),
    StructField("summary",                 StringType(),              True),
    StructField("overall_sentiment_score", FloatType(),               True),
    StructField("overall_sentiment_label", StringType(),              True),
    StructField("topics",                  ArrayType(topic_schema),   True),
    StructField("ticker_sentiment",        ArrayType(ticker_schema),  True),
])

# --- Monta as linhas ---

feed = data.get('feed', [])

rows = []
for article in feed:
    topics = [
        Row(
            topic=t.get('topic'),
            relevance_score=float(t.get('relevance_score', 0.0)),
        )
        for t in article.get('topics', [])
    ]

    tickers = [
        Row(
            ticker=t.get('ticker'),
            relevance_score=float(t.get('relevance_score', 0.0)),
            ticker_sentiment_score=float(t.get('ticker_sentiment_score', 0.0)),
            ticker_sentiment_label=t.get('ticker_sentiment_label'),
        )
        for t in article.get('ticker_sentiment', [])
    ]

    row = (
        article.get('title'),
        article.get('url'),
        article.get('time_published'),
        ', '.join(article.get('authors', [])),
        article.get('source'),
        article.get('source_domain'),
        article.get('summary'),
        article.get('overall_sentiment_score'),
        article.get('overall_sentiment_label'),
        topics,
        tickers,
    )
    rows.append(row)

# --- Cria o DataFrame PySpark ---

df = spark.createDataFrame(rows, schema=schema)

df = df.withColumn(
    "time_published",
    to_timestamp(col("time_published"), "yyyyMMdd'T'HHmmss")
)

print(f"Total de artigos: {df.count()}")
df.printSchema()

Total de artigos: 50
root
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)
 |-- time_published: timestamp (nullable = true)
 |-- authors: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_domain: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- overall_sentiment_score: float (nullable = true)
 |-- overall_sentiment_label: string (nullable = true)
 |-- topics: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- topic: string (nullable = true)
 |    |    |-- relevance_score: float (nullable = true)
 |-- ticker_sentiment: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- ticker: string (nullable = true)
 |    |    |-- relevance_score: float (nullable = true)
 |    |    |-- ticker_sentiment_score: float (nullable = true)
 |    |    |-- ticker_sentiment_label: string (nullable = true)



In [6]:
display(df.limit(2))

,title,url,time_published,authors,source,source_domain,summary,overall_sentiment_score,overall_sentiment_label,topics,ticker_sentiment
0,"Life Cycle Investment Partners Ltd Acquires Shares of 228,399 AMETEK, Inc. $AME",https://www.marketbeat.com/instant-alerts/filing-life-cycle-investment-partners-ltd-acquires-shares-of-228399-ametek-inc-ame-2026-05-24/,2026-05-24 10:15:03,MarketBeat,MarketBeat,MarketBeat,"Life Cycle Investment Partners Ltd has acquired 228,399 shares of AMETEK, Inc. (NYSE:AME) during the fourth quarter, valued at approximately $46.89 million, representing about 0.10% of AMETEK's stock. Other institutional investors have also adjusted their holdings in AMETEK, which recently reported strong quarterly earnings, beat revenue estimates, and announced a quarterly dividend of $0.34 per share. Analysts generally rate AMETEK as a ""Moderate Buy"" with an average price target of $252.33.",0.317246,Somewhat-Bullish,"[{'topic': 'earnings', 'relevance_score': 1.0}, {'topic': 'financial_markets', 'relevance_score': 0.9448980093002319}, {'topic': 'finance', 'relevance_score': 0.8296549916267395}, {'topic': 'technology', 'relevance_score': 0.7491160035133362}, {'topic': 'manufacturing', 'relevance_score': 0.6317099928855896}]","[{'ticker': 'AME', 'relevance_score': 1.0, 'ticker_sentiment_score': 0.33359000086784363, 'ticker_sentiment_label': 'Somewhat-Bullish'}]"
1,"MKS Inc. (NASDAQ:MKSI) Receives Consensus Rating of ""Moderate Buy"" from Analysts",https://www.marketbeat.com/instant-alerts/mks-inc-nasdaqmksi-receives-consensus-rating-of-moderate-buy-from-analysts-2026-05-24/,2026-05-24 07:39:48,MarketBeat,MarketBeat,MarketBeat,"MKS Inc. (NASDAQ:MKSI) has received a consensus ""Moderate Buy"" rating from analysts, with an average 12-month price target of $318.77. The company's Q1 results exceeded expectations, with EPS of $2.30 and revenue of $1.08 billion, representing a 15.2% year-over-year increase. Despite some insider selling, institutional investors like Vanguard and Dimensional Fund Advisors have increased their holdings.",0.349812,Somewhat-Bullish,"[{'topic': 'earnings', 'relevance_score': 1.0}, {'topic': 'financial_markets', 'relevance_score': 0.9005669951438904}, {'topic': 'finance', 'relevance_score': 0.820402979850769}, {'topic': 'manufacturing', 'relevance_score': 0.7111139893531799}, {'topic': 'technology', 'relevance_score': 0.6132500171661377}]","[{'ticker': 'MKSI', 'relevance_score': 1.0, 'ticker_sentiment_score': 0.32061201333999634, 'ticker_sentiment_label': 'Somewhat-Bullish'}]"


In [ ]:
df.select("overall_sentiment_label").distinct().show()

+-----------------------+
|overall_sentiment_label|
+-----------------------+
|       Somewhat-Bullish|
|                Bullish|
|                Neutral|
|       Somewhat-Bearish|
|                Bearish|
+-----------------------+



In [8]:
# Explode tickers: uma linha por ticker por artigo
df_tickers = df.select(
    "title",
    "time_published",
    "overall_sentiment_label",
    explode("ticker_sentiment").alias("t")
).select(
    "title",
    "time_published",
    "overall_sentiment_label",
    col("t.ticker").alias("ticker"),
    col("t.ticker_sentiment_score").alias("sentiment_score"),
    col("t.ticker_sentiment_label").alias("sentiment_label"),
)

df_tickers.show(10, truncate=60)

+------------------------------------------------------------+-------------------+-----------------------+------+---------------+----------------+
|                                                       title|     time_published|overall_sentiment_label|ticker|sentiment_score| sentiment_label|
+------------------------------------------------------------+-------------------+-----------------------+------+---------------+----------------+
|Life Cycle Investment Partners Ltd Acquires Shares of 228...|2026-05-24 10:15:03|       Somewhat-Bullish|   AME|        0.33359|Somewhat-Bullish|
|MKS Inc. (NASDAQ:MKSI) Receives Consensus Rating of "Mode...|2026-05-24 07:39:48|       Somewhat-Bullish|  MKSI|       0.320612|Somewhat-Bullish|
|       Covestor Ltd Lowers Stock Holdings in Flex Ltd. $FLEX|2026-05-24 07:36:50|       Somewhat-Bullish|  FLEX|       0.216199|Somewhat-Bullish|
|Honeywell Grant And Army Deal Highlight Quantum And Aeros...|2026-05-24 06:43:02|       Somewhat-Bullish|   HON|     

In [9]:
# Explode topics: uma linha por tópico por artigo
df_topics = df.select(
    "title",
    "overall_sentiment_score",
    explode("topics").alias("tp")
).select(
    "title",
    "overall_sentiment_score",
    col("tp.topic").alias("topic"),
    col("tp.relevance_score").alias("topic_relevance"),
)

df_topics.show(10, truncate=60)

+------------------------------------------------------------+-----------------------+-----------------+---------------+
|                                                       title|overall_sentiment_score|            topic|topic_relevance|
+------------------------------------------------------------+-----------------------+-----------------+---------------+
|Life Cycle Investment Partners Ltd Acquires Shares of 228...|               0.317246|         earnings|            1.0|
|Life Cycle Investment Partners Ltd Acquires Shares of 228...|               0.317246|financial_markets|       0.944898|
|Life Cycle Investment Partners Ltd Acquires Shares of 228...|               0.317246|          finance|       0.829655|
|Life Cycle Investment Partners Ltd Acquires Shares of 228...|               0.317246|       technology|       0.749116|
|Life Cycle Investment Partners Ltd Acquires Shares of 228...|               0.317246|    manufacturing|        0.63171|
|MKS Inc. (NASDAQ:MKSI) Receives